<a href="https://colab.research.google.com/github/NamSee04/CS114.P11/blob/main/CS114_P11_Tool_CreateSplit_Car.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TẠO CÁC TẬP DỮ LIỆU TRAIN, TEST (SPLITS)

1. Yêu cầu chung: Tạo ra các splits, mỗi split tương ứng với một tập dữ liệu train - test
  + Bài học lý thuyết (để trả lời cho các câu hỏi như vì sao phải cần việc này, thực hiện việc này như thế nào):
    - https://www.kaggle.com/code/satishgunjal/tutorial-k-fold-cross-validation
    - https://machinelearningmastery.com/training-validation-test-split-and-cross-validation-done-right/
    - https://machinelearningmastery.com/how-to-configure-k-fold-cross-validation/

  <IMG SRC = 'https://raw.githubusercontent.com/satishgunjal/images/master/KFold_Cross_Validation.png'>
2. Yêu cầu cụ thể:
- Input:
    + Thư mục cha chứa các thư mục con - mỗi thư mục con tương ứng với tên của từng hiệu xe (Honda, Suzuki, VinFast, Yamaha, Others). Ví dụ: https://drive.google.com/drive/u/1/folders/1Uj0V9URNHpzSHeXHSB89AoGCjGki8Yra
    + Các ảnh được đặt tên theo quy ước: các tập tin ảnh theo quy ước
    + Số splits NumSplits - mặc định NumSplits=5 (tương đương 5-fold CV)
- Output:
    + File CarDataset.csv - Tập tin chứa tất cả ảnh của dataset
      - Chương trình sẽ scan qua cây thư mục để tìm tất cả các ảnh (chỉ chọn các ảnh có định dạng + phần mở rộng là .jpg)
      - Mỗi dòng sẽ có các thông tin cách nhau bằng dấu phẩy, theo quy ước: ImageFullPath, CategoryID
            - ImageFullPath ở dạng <Thư mục Hiệu xe>/<Tên ảnh>. Ví dụ: Honda/2024123.Honda.1.jpg.
            - CategoryID là số nguyên thuộc [0..5] theo quy ước
              - 0: Others
              - 1: Honda
              - 2: Suzuki
              - 3: Yamaha
              - 4: VinFast     
    + File CarDataset-Splits-[1..5]-[Train/Test].csv - Phân chia thành các splits, mỗi split gồm các ảnh được chia thành thành 2 tập Train - Test
      + Chương trình sẽ đọc dữ liệu từ file CarDataset.csv, sau đó với mỗi hiệu xe, chia ngẫu nhiên thành 5 tập dữ liệu. Lưu ý là phải chia theo hiệu xe, để đảm bảo dữ liệu Train/Test có dữ liệu của các hiệu xe.
      + Từ 5 tập dữ liệu chia ngẫu nhiên theo các hiệu xe Xij (i là thứ tự tập dữ liệu, j là CategoryID), gom lại thành 5 tập dữ liệu lớn hơn Xi, sao cho mỗi tập dữ liệu này chứa đủ dữ liệu của tất cả các hiệu xe. Tức là Xi = Union(Xij)       
      + Từ 5 tập dữ liệu Xi này (tương ứng với FOLDi ở trong hình vẽ trên), tạo ra 5 splits Split-i
      + Với mỗi bộ dữ liệu Split-i, ghi xuống thành 2 tập tin tương ứng với Train, Test. Ví dụ Split-1 thì ghi thành tập tin  CarDataset-Splits-1-Train.csv và CarDataset-Splits-1-Test.csv. Tập Train gồm 4 bộ, Test gồm bộ còn lại. Ví dụ,
          + Split-1: tập Train sẽ gồm X2, X3, X4, X5, tập Test là X1
          + Split-5: tập Train sẽ gồm X1, X2, X3, X4, tập Test là X5
      + Mỗi dòng sẽ có các thông tin cách nhau bằng dấu phẩy, theo quy ước: ImageFullPath, CategoryID
- Lưu ý:
  - Nên viết thêm các cell
    - Hiển thị danh sách các tên tập tin ảnh trong từng Split-Train/Test,
    - Thống kê các ảnh cho từng CategoryID trong mỗi Split-Train/Test
  - Cần có chú thích
3. Nộp bài: SV share notebook. Các bài nộp sớm sẽ được full điểm. Deadline: TBA
4. Bài làm đạt yêu cầu sẽ được paste vào notebook với ghi nhận đóng góp từ tác giả.

## Thông tin của tác giả, ngày cập nhật

In [33]:
from google.colab import drive
import os

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import os
import csv
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Hỗ trợ các định dạng file hợp lệ
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
LABEL = {
    'Others' : 0,
    'Honda' : 1,
    'Suzuki' : 2,
    'KIA' : 3,
    'VinFast' : 4,
    'Mazda' : 5,
    'Mitsubishi' : 6,
    'Toyota' : 7,
    'Hyundai' : 8,
}

In [35]:
def process_directory(root_dir):
    """Duyệt thư mục và thống kê số lượng ảnh."""
    data = []
    labels = []

    for car_brand in os.listdir(root_dir):
        brand_dir = os.path.join(root_dir, car_brand)
        if not os.path.isdir(brand_dir) or car_brand not in LABEL:
            continue
        for file_name in os.listdir(brand_dir):
            file_path = os.path.join(brand_dir, file_name)
            ext = os.path.splitext(file_name)[-1].lower()
            if ext in VALID_EXTENSIONS:
                data.append(file_path)
                labels.append(LABEL[car_brand])
    return np.array(data), np.array(labels)

In [36]:
def create_stratified_splits(data, labels, n_splits=5):
    """Chia dữ liệu thành các tập Stratified Train/Test."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = []
    for train_idx, test_idx in skf.split(data, labels):
        train_data = [[data[i], labels[i]] for i in train_idx]
        test_data = [[data[i], labels[i]] for i in test_idx]
        splits.append((train_data, test_data))
    return splits

def save_splits_to_csv(splits, output_dir):
    """Ghi các Split-i vào file CSV."""
    os.makedirs(output_dir, exist_ok=True)
    for i, (train_data, test_data) in enumerate(splits):
        train_file = os.path.join(output_dir, f"CarDataset-Split-{i+1}-Train.csv")
        test_file = os.path.join(output_dir, f"CarDataset-Split-{i+1}-Test.csv")

        with open(train_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(train_data)

        with open(test_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(test_data)

In [37]:
root_dir = '/content/drive/MyDrive/Public'
output_dir = '/content/drive/MyDrive/CarDatasetSplits'

data, labels = process_directory(root_dir)
splits = create_stratified_splits(data, labels)
save_splits_to_csv(splits, output_dir)

print("Data splitting and CSV creation completed!")

Data splitting and CSV creation completed!
